# Player Performance Rating Prediction (Regression Pipeline)
## Track 1: Multi-Model Regression, Hyperparameter Tuning & Diagnostics

In this pipeline, we build, evaluate, and compare 10 regression algorithms to predict player `performance_score`. All preprocessing (data cleaning, missing value imputation, domain feature engineering) has been executed leak-free in `notebooks/preprocessing.ipynb` and loaded directly from `data/df_preprocessed_regression.csv`.

In [1]:
# Import analytical, regression, and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = "Segoe UI"
plt.rcParams["figure.dpi"] = 120


## Section 1: Data Preparation & Leak-Free Feature Scaling

We load the preprocessed regression dataset, select predictor metrics, partition into 80% train and 20% test splits, and fit the `StandardScaler` strictly on the training set to eliminate data leakage.

In [2]:
# Load preprocessed dataset
df_reg = pd.read_csv("../data/df_preprocessed_regression.csv")

# Select feature columns and target
feature_cols = [
    "age", "height_cm", "weight_kg", "minutes_played", "goals", "assists",
    "shots", "shots_on_target", "expected_goals_xg", "expected_assists_xa",
    "key_passes", "pass_accuracy", "tackles", "interceptions", "clearances",
    "blocks", "recoveries", "fouls_committed", "distance_covered_km",
    "sprint_distance_km", "top_speed_kmh", "stamina_score", "possession_impact",
    "creativity_score", "consistency_score", "pressure_resistance",
    "goal_contribution_rate", "work_rate_intensity", "passing_efficiency"
]

X = df_reg[feature_cols]
y = df_reg["performance_score"]

# Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Leak-free Scaling (Fit ONLY on X_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Preprocessed Dataset shape: {df_reg.shape}")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print("[OK] StandardScaler fitted strictly on training data (Zero Data Leakage).")


Preprocessed Dataset shape: (31558, 79)
X_train shape: (25246, 29), X_test shape: (6312, 29)
[OK] StandardScaler fitted strictly on training data (Zero Data Leakage).


## Section 2: Core Regression Algorithms & Multi-Model Evaluation

We train and evaluate 10 standard regression algorithms across linear, non-linear, tree-based, ensemble, and kernel methods using $R^2$, RMSE, and MAE metrics.

In [ ]:
# Define 10 Regression Models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01),
    "ElasticNet Regression": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=8),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(n_estimators=50, random_state=42),
    "Support Vector Regressor": SVR(C=1.0, epsilon=0.1)
}

results_list = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    
    results_list.append({
        "Model": name,
        "R2 Score": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MSE": mse
    })
    fitted_models[name] = (model, y_pred)
    print(f"{name:25s} | R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")


Linear Regression         | R2: 0.2386 | RMSE: 6.9560 | MAE: 5.5656
Ridge Regression          | R2: 0.2386 | RMSE: 6.9560 | MAE: 5.5656
Lasso Regression          | R2: 0.2384 | RMSE: 6.9570 | MAE: 5.5662
ElasticNet Regression     | R2: 0.2385 | RMSE: 6.9568 | MAE: 5.5661
K-Nearest Neighbors       | R2: 0.0829 | RMSE: 7.6344 | MAE: 6.0687
Decision Tree             | R2: 0.1900 | RMSE: 7.1747 | MAE: 5.7296
Random Forest             | R2: 0.2196 | RMSE: 7.0424 | MAE: 5.6193
Gradient Boosting         | R2: 0.2421 | RMSE: 6.9403 | MAE: 5.5489
AdaBoost Regressor        | R2: 0.2189 | RMSE: 7.0457 | MAE: 5.6334


### Model Performance Comparison Table
Comparative summary table of all 10 algorithms ranked by test set $R^2$ performance.

In [ ]:
# Summary Comparison Table
df_results = pd.DataFrame(results_list).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
print(df_results.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x="R2 Score", y="Model", palette="viridis")
plt.title("Regression Model Performance ($R^2$ Score Comparison)", fontsize=12, fontweight="bold")
plt.xlabel("R^2 Score")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()


## Section 3: 5-Fold Cross-Validation & Hyperparameter Tuning

We perform 5-fold cross-validation on the top performing models and apply `GridSearchCV` to optimize hyperparameters.

In [ ]:
# 5-Fold Cross-Validation on Top Models
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_gbm = cross_val_score(GradientBoostingRegressor(random_state=42), X_train_scaled, y_train, cv=kf, scoring="r2")
cv_rf = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42), X_train_scaled, y_train, cv=kf, scoring="r2")

print(f"Gradient Boosting 5-Fold CV R2: {cv_gbm.mean():.4f} (+/- {cv_gbm.std():.4f})")
print(f"Random Forest 5-Fold CV R2:     {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})")

# GridSearchCV for Gradient Boosting
param_grid = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5]
}
grid_search = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid, cv=3, scoring="r2", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_gbm = grid_search.best_estimator_
best_pred = best_gbm.predict(X_test_scaled)
print("Best Hyperparameters:", grid_search.best_params_)
print(f"Tuned Test R2: {r2_score(y_test, best_pred):.4f}")


## Section 4: Diagnostic Plots & Feature Importances

Visual evaluation of residual distribution, predicted vs actual target values, and key feature importance weights.

In [ ]:
# Diagnostic Plots: Residual Plot & Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residuals = y_test - best_pred
axes[0].scatter(best_pred, residuals, alpha=0.4, color="teal")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residual Plot (Best Model)", fontweight="bold")
axes[0].set_xlabel("Predicted Performance Score")
axes[0].set_ylabel("Residuals (Actual - Predicted)")

axes[1].scatter(y_test, best_pred, alpha=0.4, color="purple")
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[1].set_title("Predicted vs Actual Target Values", fontweight="bold")
axes[1].set_xlabel("Actual Performance Score")
axes[1].set_ylabel("Predicted Performance Score")

plt.tight_layout()
plt.show()

# Feature Importance Plot
importances = best_gbm.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette="magma")
plt.title("Top 10 Feature Importances (Gradient Boosting Regressor)", fontsize=12, fontweight="bold")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()
